<a href="https://colab.research.google.com/github/Rinosa123/Bilingual-Enterprise-RAG-Copilot/blob/main/notebooks/01_multilingual_dense_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri Aug  7 11:33:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Clone GitHub repository**

In [1]:
%cd /content

!git clone https://github.com/Rinosa123/Bilingual-Enterprise-RAG-Copilot.git

%cd /content/Bilingual-Enterprise-RAG-Copilot

/content
fatal: destination path 'Bilingual-Enterprise-RAG-Copilot' already exists and is not an empty directory.
/content/Bilingual-Enterprise-RAG-Copilot


In [2]:
!pip -q install sentence-transformers

### **Load the documents and embedding model**

In [3]:
from pathlib import Path

import numpy as np
import sentence_transformers
import torch
from sentence_transformers import SentenceTransformer

from src.ingestion.chunker import chunk_documents
from src.ingestion.text_loader import load_text_documents


PROJECT_ROOT = Path.cwd()
DOCUMENT_DIRECTORY = PROJECT_ROOT / "data" / "sample_docs"

MODEL_NAME = "intfloat/multilingual-e5-small"


print("Sentence Transformers:", sentence_transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


documents = load_text_documents(DOCUMENT_DIRECTORY)
chunks = chunk_documents(documents)

print("Documents:", len(documents))
print("Chunks:", len(chunks))


model = SentenceTransformer(MODEL_NAME)

print("Model:", MODEL_NAME)
print("Device:", model.device)

Sentence Transformers: 5.6.0
PyTorch: 2.11.0+cu128
CUDA available: True
Documents: 2
Chunks: 10


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model: intfloat/multilingual-e5-small
Device: cuda:0


### **Create document embeddings**

In [4]:
passage_texts = [
    f"passage: {chunk.section}\n{chunk.text}"
    for chunk in chunks
]

passage_embeddings = model.encode(
    passage_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

print("Passage embedding shape:", passage_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Passage embedding shape: (10, 384)


### **Evaluate cross-language retrieval**

In [5]:
EVALUATION_QUERIES = (
    (
        "English question -> English document",
        "How many annual leave days do full-time employees receive?",
        "HR-EN-001-CH-003",
    ),
    (
        "Arabic question -> Arabic document",
        "ما الحد الأقصى لتكلفة الفندق؟",
        "HR-AR-001-CH-003",
    ),
    (
        "Arabic question -> English document",
        "كم عدد أيام الإجازة السنوية للموظف؟",
        "HR-EN-001-CH-003",
    ),
    (
        "English question -> Arabic document",
        "When must an expense claim be submitted?",
        "HR-AR-001-CH-002",
    ),
)


query_texts = [
    f"query: {query}"
    for _, query, _ in EVALUATION_QUERIES
]

query_embeddings = model.encode(
    query_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

similarity_matrix = query_embeddings @ passage_embeddings.T

successful_queries = 0


for query_index, (
    label,
    question,
    expected_chunk_id,
) in enumerate(EVALUATION_QUERIES):

    scores = similarity_matrix[query_index]
    top_chunk_index = int(np.argmax(scores))

    retrieved_chunk = chunks[top_chunk_index]
    similarity_score = float(scores[top_chunk_index])

    passed = retrieved_chunk.chunk_id == expected_chunk_id

    if passed:
        successful_queries += 1

    outcome = "PASS" if passed else "MISS"

    print("=" * 80)
    print("Test:", label)
    print("Question:", question)
    print("Expected:", expected_chunk_id)
    print("Retrieved:", retrieved_chunk.chunk_id)
    print("Section:", retrieved_chunk.section)
    print(f"Similarity score: {similarity_score:.4f}")
    print("Result:", outcome)


total_queries = len(EVALUATION_QUERIES)
accuracy = successful_queries / total_queries

print("=" * 80)
print(
    f"Dense retrieval accuracy: "
    f"{successful_queries}/{total_queries} "
    f"({accuracy:.0%})"
)
print("BM25 baseline accuracy: 2/4 (50%)")

Test: English question -> English document
Question: How many annual leave days do full-time employees receive?
Expected: HR-EN-001-CH-003
Retrieved: HR-EN-001-CH-003
Section: 2. Annual Leave
Similarity score: 0.8941
Result: PASS
Test: Arabic question -> Arabic document
Question: ما الحد الأقصى لتكلفة الفندق؟
Expected: HR-AR-001-CH-003
Retrieved: HR-AR-001-CH-003
Section: 2. السفر في مهام العمل
Similarity score: 0.8903
Result: PASS
Test: Arabic question -> English document
Question: كم عدد أيام الإجازة السنوية للموظف؟
Expected: HR-EN-001-CH-003
Retrieved: HR-AR-001-CH-004
Section: 3. الإجازة المرضية
Similarity score: 0.8559
Result: MISS
Test: English question -> Arabic document
Question: When must an expense claim be submitted?
Expected: HR-AR-001-CH-002
Retrieved: HR-AR-001-CH-002
Section: 1. مطالبات المصروفات
Similarity score: 0.7884
Result: PASS
Dense retrieval accuracy: 3/4 (75%)
BM25 baseline accuracy: 2/4 (50%)


In [6]:
chunk_index_by_id = {
    chunk.chunk_id: index
    for index, chunk in enumerate(chunks)
}

top_1_hits = 0
top_3_hits = 0
reciprocal_ranks = []


for query_index, (
    label,
    question,
    expected_chunk_id,
) in enumerate(EVALUATION_QUERIES):

    scores = similarity_matrix[query_index]
    ranked_indices = np.argsort(-scores)

    expected_chunk_index = chunk_index_by_id[
        expected_chunk_id
    ]

    expected_rank = (
        int(
            np.where(
                ranked_indices == expected_chunk_index
            )[0][0]
        )
        + 1
    )

    top_1_hits += int(expected_rank == 1)
    top_3_hits += int(expected_rank <= 3)
    reciprocal_ranks.append(1 / expected_rank)

    print("=" * 80)
    print("Test:", label)
    print("Expected chunk:", expected_chunk_id)
    print("Expected rank:", expected_rank)
    print("Top three retrieved chunks:")

    for rank, chunk_index in enumerate(
        ranked_indices[:3],
        start=1,
    ):
        retrieved_chunk = chunks[int(chunk_index)]

        print(
            f"  {rank}. {retrieved_chunk.chunk_id} | "
            f"score={scores[int(chunk_index)]:.4f} | "
            f"section={retrieved_chunk.section}"
        )


number_of_queries = len(EVALUATION_QUERIES)

top_1_accuracy = top_1_hits / number_of_queries
hit_at_3 = top_3_hits / number_of_queries
mean_reciprocal_rank = sum(reciprocal_ranks) / number_of_queries


print("=" * 80)
print(f"Top-1 accuracy: {top_1_accuracy:.0%}")
print(f"Hit@3: {hit_at_3:.0%}")
print(f"MRR: {mean_reciprocal_rank:.4f}")

Test: English question -> English document
Expected chunk: HR-EN-001-CH-003
Expected rank: 1
Top three retrieved chunks:
  1. HR-EN-001-CH-003 | score=0.8941 | section=2. Annual Leave
  2. HR-EN-001-CH-002 | score=0.7900 | section=1. Working Hours
  3. HR-EN-001-CH-004 | score=0.7808 | section=3. Remote Work
Test: Arabic question -> Arabic document
Expected chunk: HR-AR-001-CH-003
Expected rank: 1
Top three retrieved chunks:
  1. HR-AR-001-CH-003 | score=0.8903 | section=2. السفر في مهام العمل
  2. HR-AR-001-CH-002 | score=0.7946 | section=1. مطالبات المصروفات
  3. HR-AR-001-CH-005 | score=0.7727 | section=4. التدريب والتطوير المهني
Test: Arabic question -> English document
Expected chunk: HR-EN-001-CH-003
Expected rank: 4
Top three retrieved chunks:
  1. HR-AR-001-CH-004 | score=0.8559 | section=3. الإجازة المرضية
  2. HR-AR-001-CH-003 | score=0.8272 | section=2. السفر في مهام العمل
  3. HR-AR-001-CH-005 | score=0.8232 | section=4. التدريب والتطوير المهني
Test: English question -> Ara